# Experiment 3: model-family comparison

This experiment isolates model-family effects. Every model receives exactly the Experiment 2 feature set (`Age`, `SibSp`, `Parch`, `Fare`, `Pclass`, `Sex`, `Embarked`, engineered `Title`, and engineered `IsAlone`) and the same five shuffled stratified folds. No leaderboard result is used for selection and no hyperparameter search is performed.

In [1]:
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (GradientBoostingClassifier, HistGradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data' / 'raw'
SUBMISSION_DIR = ROOT / 'submissions'
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'gender_submission.csv')
X_raw = train.drop(columns='Survived')
y = train['Survived']
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(X_raw, y))
print(f'Train: {train.shape}; test: {test.shape}; fixed folds: {len(folds)}')

Train: (891, 12); test: (418, 11); fixed folds: 5


## Fixed features and leakage-safe preprocessing

`Title` uses the same four common categories plus pooled `Rare` category as Experiment 2. `IsAlone` is the same indicator derived from `SibSp + Parch`. Numeric values are median-imputed; categorical values are mode-imputed and one-hot encoded with unknown categories ignored. All learned preprocessing is inside each model pipeline and is refit within every training fold. Logistic regression retains numeric standardization. Tree models omit unnecessary scaling and use dense one-hot output for estimator compatibility.

In [2]:
NUMERIC = ['Age', 'SibSp', 'Parch', 'Fare']
CATEGORICAL = ['Pclass', 'Sex', 'Embarked', 'Title', 'IsAlone']
COMMON_TITLES = {'Mr', 'Miss', 'Mrs', 'Master'}

def engineer_features(df):
    df = df.copy()
    title = df['Name'].str.extract(r',\s*([^.]*)\.', expand=False).str.strip()
    df['Title'] = title.where(title.isin(COMMON_TITLES), 'Rare')
    df['IsAlone'] = ((df['SibSp'] + df['Parch']) == 0).astype(int)
    return df

X = engineer_features(X_raw)
X_test = engineer_features(test)

def make_preprocessor(scale_numeric):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer([
        ('numeric', Pipeline(numeric_steps), NUMERIC),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), CATEGORICAL),
    ])

def make_pipeline(estimator, scale_numeric=False):
    return Pipeline([
        ('preprocessing', make_preprocessor(scale_numeric)),
        ('model', estimator),
    ])

## Models and tradeoffs

- **LogisticRegression (control):** stable, fast, and interpretable, but its additive linear decision boundary cannot naturally express nonlinear thresholds or interactions.
- **RandomForestClassifier:** captures nonlinearities and interactions and is relatively robust, but can overfit small tabular datasets and is less directly interpretable. We use 300 trees to reduce Monte Carlo variability while leaving other behavior at defaults.
- **HistGradientBoostingClassifier:** efficient boosting with nonlinear interactions and regularization-friendly defaults, but may be less stable on a dataset this small and loses native categorical structure after one-hot encoding.
- **GradientBoostingClassifier:** sequential shallow trees can capture useful nonlinear structure on small tabular data, but boosting is more sensitive to noise and hyperparameters than logistic regression.

Fit times below are approximate wall-clock measurements and are environment-dependent.

In [3]:
model_specs = {
    'LogisticRegression': (LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), True),
    'RandomForestClassifier': (RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1), False),
    'HistGradientBoostingClassifier': (HistGradientBoostingClassifier(random_state=RANDOM_STATE), False),
    'GradientBoostingClassifier': (GradientBoostingClassifier(random_state=RANDOM_STATE), False),
}

all_scores, all_times = {}, {}
for name, (estimator, scale_numeric) in model_specs.items():
    fold_scores, fold_times = [], []
    for train_idx, valid_idx in folds:
        pipeline = make_pipeline(estimator, scale_numeric)
        started = perf_counter()
        pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
        fold_times.append(perf_counter() - started)
        fold_scores.append(accuracy_score(y.iloc[valid_idx], pipeline.predict(X.iloc[valid_idx])))
    all_scores[name] = np.array(fold_scores)
    all_times[name] = np.array(fold_times)

control = all_scores['LogisticRegression']
assert np.isclose(control.mean(), 0.8305, atol=5e-5)
rows = []
for name in model_specs:
    scores = all_scores[name]
    rows.append({
        'Model': name,
        **{f'Fold {i}': score for i, score in enumerate(scores, 1)},
        'Mean': scores.mean(),
        'Std': scores.std(),
        'Delta vs logistic': scores.mean() - control.mean(),
        'Mean fit seconds/fold': all_times[name].mean(),
    })
results = pd.DataFrame(rows).set_index('Model')
display(results.style.format('{:.4f}'))

,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean,Std,Delta vs logistic,Mean fit seconds/fold
Model,,,,,,,,,
LogisticRegression,0.8380,0.8146,0.8371,0.8315,0.8315,0.8305,0.0084,0.0000,0.0066
RandomForestClassifier,0.8268,0.7978,0.7809,0.8258,0.8202,0.8103,0.0181,-0.0202,0.1743
HistGradientBoostingClassifier,0.8492,0.8315,0.7978,0.8202,0.8596,0.8316,0.0217,0.0011,0.4579
GradientBoostingClassifier,0.8547,0.8483,0.8258,0.8315,0.8371,0.8395,0.0107,0.0090,0.0553


## Selection decision

Gradient boosting is the only nonlinear model with a practically noticeable mean gain over the logistic control: about 0.9 percentage points. It wins three paired folds, ties one, and loses one. Its standard deviation is only modestly higher than logistic regression's. This is reasonably consistent evidence for an Experiment 3 candidate, but not decisive evidence: the dataset is small, the gain corresponds to only a handful of out-of-fold predictions, and one fold regresses.

Random forest is clearly worse and less stable. Histogram gradient boosting gains only about 0.1 percentage points while substantially increasing fold variability, so it does not convincingly beat the control. Selection is based exclusively on these local folds.

In [4]:
selected_name = 'GradientBoostingClassifier'
selected_scores = all_scores[selected_name]
paired_delta = selected_scores - control
print('Selected model:', selected_name)
print('Fold scores:', np.round(selected_scores, 4).tolist())
print(f'Mean ± std: {selected_scores.mean():.4f} ± {selected_scores.std():.4f}')
print(f'Delta vs logistic: {selected_scores.mean() - control.mean():+.4f}')
print('Paired fold deltas:', np.round(paired_delta, 4).tolist())
print(f'Wins / ties / losses: {(paired_delta > 0).sum()} / {(paired_delta == 0).sum()} / {(paired_delta < 0).sum()}')

Selected model: GradientBoostingClassifier
Fold scores: [0.8547, 0.8483, 0.8258, 0.8315, 0.8371]
Mean ± std: 0.8395 ± 0.0107
Delta vs logistic: +0.0090
Paired fold deltas: [0.0168, 0.0337, -0.0112, 0.0, 0.0056]
Wins / ties / losses: 3 / 1 / 1


## Fit selected model and verify submission

Because a different model family was selected, fit it on all training rows and create `submission_03_model.csv`. Verification checks schema, row count, passenger ordering, integer dtypes, missingness, unique IDs, binary predictions, and exact disk round-trip. The file is prepared for a possible Kaggle experiment but is not submitted here.

In [5]:
selected_model = make_pipeline(GradientBoostingClassifier(random_state=RANDOM_STATE), scale_numeric=False)
selected_model.fit(X, y)
predictions = selected_model.predict(X_test).astype(int)
submission = pd.DataFrame({'PassengerId': test['PassengerId'].astype(int), 'Survived': predictions})

assert submission.columns.tolist() == sample_submission.columns.tolist() == ['PassengerId', 'Survived']
assert len(submission) == len(test) == len(sample_submission)
assert submission['PassengerId'].equals(test['PassengerId'].astype(int))
assert submission['PassengerId'].equals(sample_submission['PassengerId'].astype(int))
assert pd.api.types.is_integer_dtype(submission['PassengerId'])
assert pd.api.types.is_integer_dtype(submission['Survived'])
assert not submission.isna().any().any()
assert submission['PassengerId'].is_unique
assert set(submission['Survived'].unique()).issubset({0, 1})

SUBMISSION_DIR.mkdir(exist_ok=True)
submission_path = SUBMISSION_DIR / 'submission_03_model.csv'
submission.to_csv(submission_path, index=False)
pd.testing.assert_frame_equal(pd.read_csv(submission_path), submission)
print(f'Wrote and verified {submission_path.relative_to(ROOT)} ({len(submission)} rows)')
print(submission['Survived'].value_counts().sort_index())
display(submission.head())

Wrote and verified submissions/submission_03_model.csv (418 rows)
Survived
0    274
1    144
Name: count, dtype: int64


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
